# Epithelial Cell Annotation Pipeline
## Cluster + GEP Combined Analysis

**Goal**: Annotate epithelial cell subtypes using:
- BBKNN batch-corrected clustering
- Harmony GEPs (Gene Expression Programs)
- Comprehensive marker gene list

**Strategy**:
1. Data overview and quality check
2. GEP-Cluster association heatmap ⭐
3. Marker gene dotplot
4. Focused analysis on ambiguous clusters
5. Cross-tissue validation

---

## 📦 Setup and Data Loading

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting parameters
sc.set_figure_params(dpi=100, frameon=False, figsize=(6, 6), facecolor='white')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

print("✓ Libraries loaded")
print(f"scanpy version: {sc.__version__}")

In [ ]:
# ============================================================================
# CONFIGURATION - Modify these paths
# ============================================================================

DATA_PATH = "/home/h2048/data/py/1202/bbknn_celltype_analysis/Epithelial/output_complete_pipeline/checkpoint_complete_with_geps.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1207/bbknn_celltype_analysis/Epithelial/annotation_results"

# Create output directory
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Key column names
CLUSTER_KEY = 'leiden_bbknn'  # Using BBKNN result
TISSUE_KEY = 'tissue'  # Tissue origin
BATCH_KEY = 'dataset'  # Batch info

# GEP source (using Harmony-corrected GEPs)
GEP_SUFFIX = 'harmony'  # or 'nocorr'
N_GEPS = 15

print(f"✓ Configuration set")
print(f"   Data: {DATA_PATH}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Cluster key: {CLUSTER_KEY}")
print(f"   GEP suffix: {GEP_SUFFIX}")

In [ ]:
# Load data
print("Loading data...")
adata = sc.read_h5ad(DATA_PATH)

print(f"\n✓ Data loaded")
print(f"   Cells: {adata.n_obs:,}")
print(f"   Genes: {adata.n_vars:,}")
print(f"\nAvailable obs columns:")
print(adata.obs.columns.tolist())

In [ ]:
# Quick data inspection
print("Data Overview:")
print(f"\n1. Cluster distribution:")
print(adata.obs[CLUSTER_KEY].value_counts().sort_index())

print(f"\n2. Tissue distribution:")
print(adata.obs[TISSUE_KEY].value_counts())

print(f"\n3. Batch distribution:")
print(adata.obs[BATCH_KEY].value_counts())

# Check GEP columns
gep_cols = [col for col in adata.obs.columns if col.startswith('GEP_') and col.endswith(f'_{GEP_SUFFIX}')]
print(f"\n4. Found {len(gep_cols)} GEP columns")
print(f"   {gep_cols[:3]} ... {gep_cols[-1]}")

## 🧬 Define Comprehensive Marker Genes

Based on the detailed epithelial cell subtype classification

In [ ]:
# Comprehensive marker gene dictionary
MARKER_GENES = {
    # ========== Alveolar cells ==========
    'AT1': ['AGER', 'CAV1', 'MYL9', 'SFTA2', 'CLIC3', 'SPOCK2', 'ANXA3', 'RTKN2', 'TIMP3', 'TNNC1'],
    'AT2': ['SFTPB', 'SFTPC', 'SFTPA1', 'SFTPA2', 'LAMP3', 'LRRK2', 'TFPI', 'MFSD2A', 'SERPINA1'],
    'AT2_proliferating': ['STMN1', 'SFTPA2', 'SFTPA1', 'TYMS', 'TK1', 'PTTG1', 'KIAA0101', 'TOP2A', 'CENPW', 'DTYMK'],
    'Transitional_Club_AT2': ['SCGB3A2', 'MGP', 'C16orf89', 'SFTA1P', 'VIM', 'SFTPA2', 'CAV1', 'ICAM1', 'SUSD2'],
    
    # ========== Basal cells ==========
    'Basal_resting': ['KRT15', 'KRT17', 'KRT5', 'DST', 'DLK2', 'IL33', 'FHL2', 'PTPRZ1'],
    'Suprabasal': ['KRT5', 'KRT17', 'SERPINB4', 'SERPINB13', 'KRT6A', 'CLCA2', 'LY6D', 'PPP1R14B', 'AKR1C3', 'IGFBP3'],
    'Basal_general': ['TP63', 'KRT5', 'KRT14'],  # General basal markers
    
    # ========== Ciliated cells ==========
    'Deuterosomal': ['CCNO', 'CDC20B', 'ZMYND10', 'KIF9', 'FOXJ1', 'HES6', 'CEP78', 'TMEM106C', 'CCDC67', 'KDELC2'],
    'Multiciliated_nasal': ['C20orf85', 'C9orf24', 'RSPH1', 'PIFO', 'RP11-356K23.1', 'CCDC80', 'PROM1', 'OMG', 'DIAPH2', 'C15orf48'],
    'Multiciliated_non_nasal': ['C20orf85', 'CAPS', 'C9orf24', 'RSPH1', 'FAM183A', 'MS4A8', 'TFF3', 'IGFBP5', 'CFAP43', 'C2orf40'],
    'Ciliated_general': ['FOXJ1', 'RSPH1', 'PIFO'],  # General ciliated markers
    
    # ========== Club/Secretory cells ==========
    'Club_non_nasal': ['TSPAN8', 'CYP2F1', 'TFF3', 'TGM2', 'MUC5B', 'CXCL6', 'C16orf89', 'HES4', 'RHOV', 'KIAA1324'],
    'Club_nasal': ['ASRGL1', 'LYPD2', 'UGT2A1', 'TFCP2L1', 'LY6D', 'TPD52L1', 'SORD', 'PI3'],
    'Secretory_general': ['SCGB1A1', 'SCGB3A2', 'SERPINB3'],  # General secretory markers
    
    # ========== Goblet cells ==========
    'Goblet_nasal': ['LYPD2', 'PI3', 'CEACAM5', 'LYNX1', 'MUC5AC', 'MUC16', 'C15orf48', 'BPIFA1', 'CCDC80', 'DHRS9'],
    'Goblet_bronchial': ['MUC5B', 'RARRES1', 'SAA1', 'ANKRD36C', 'SAA2', 'LYZ', 'PLCG2', 'FCGBP', 'RIMS1', 'MUC5AC'],
    'Goblet_subegmental': ['TSPAN8', 'MUC5B', 'C16orf89', 'MTRNR2L10', 'CLCA2', 'CFD', 'KIAA1324', 'LTF', 'TMEM45A', 'FCGBP'],
    'Goblet_general': ['MUC5AC', 'SPDEF', 'LYPD2', 'ITLN1'],  # General goblet markers
    
    # ========== SMG cells ==========
    'SMG_serous_nasal': ['LYZ', 'ZG16B', 'AZGP1', 'LTF', 'STATH', 'PIP', 'CLDN10', 'GJC3', 'ODAM', 'S100A1'],
    'SMG_serous_bronchial': ['LYZ', 'LTF', 'PRR4', 'AZGP1', 'S100A1', 'APIP', 'RP11-1143G9.4', 'PRB3', 'AC078941.1', 'C6orf58'],
    'SMG_mucous': ['MUC5B', 'BPIFB2', 'AZGP1', 'FCGBP', 'NKX3-1', 'TSPAN8', 'TFF1', 'DEFB1', 'HMGCS2', 'CRYM'],
    'SMG_duct': ['RARRES1', 'TCN1', 'SAA1', 'MIA', 'DMBT1', 'SAA2', 'RHOV', 'MMP7', 'ALDH1A3', 'ANKRD36C'],
    
    # ========== Rare cells ==========
    'Ionocyte': ['RARRES2', 'TMEM61', 'ASCL3', 'SCNN1B', 'STAP1', 'ATP6V1A', 'CFTR', 'HEPACAM2', 'CLCNKB', 'FOXI1'],
    'Tuft': ['STMN1', 'MARCKSL1', 'RASSF6', 'CRYM', 'HES6', 'KIT', 'AZGP1', 'HOMER3', 'NREP', 'LRMP'],
    'Neuroendocrine': ['PCSK1N', 'GRP', 'CPE', 'ASCL1', 'CHGA', 'SCG2', 'SCG5', 'SYT1', 'SCG3', 'SCGN'],
    
    # ========== Proliferation ==========
    'Proliferating': ['MKI67', 'TOP2A', 'TK1', 'CENPW', 'STMN1'],
}

# Create a flattened list of all markers (for initial screening)
ALL_MARKERS = sorted(set([gene for genes in MARKER_GENES.values() for gene in genes]))

# Filter markers present in the dataset
available_markers = [m for m in ALL_MARKERS if m in adata.var_names]

print(f"Total unique markers defined: {len(ALL_MARKERS)}")
print(f"Available in dataset: {len(available_markers)} ({len(available_markers)/len(ALL_MARKERS)*100:.1f}%)")
print(f"\nMissing markers: {len(ALL_MARKERS) - len(available_markers)}")

missing = [m for m in ALL_MARKERS if m not in adata.var_names]
if len(missing) > 0 and len(missing) < 50:
    print(f"Missing genes: {', '.join(missing[:20])}...")

---
## 📊 Round 1: Overview Visualization

### 1.1 UMAP Four-Panel Overview

In [ ]:
# UMAP overview
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Panel 1: Batch
sc.pl.umap(adata, color=BATCH_KEY, ax=axes[0, 0], show=False, 
           title='Batch Distribution', frameon=False, legend_fontsize=10)

# Panel 2: Clusters
sc.pl.umap(adata, color=CLUSTER_KEY, ax=axes[0, 1], show=False,
           title='BBKNN Clusters', frameon=False, 
           legend_loc='on data', legend_fontsize=8)

# Panel 3: Tissue
sc.pl.umap(adata, color=TISSUE_KEY, ax=axes[1, 0], show=False,
           title='Tissue Origin', frameon=False, legend_fontsize=10)

# Panel 4: QC metric
if 'n_counts' in adata.obs.columns:
    sc.pl.umap(adata, color='n_counts', ax=axes[1, 1], show=False,
               title='UMI Counts', frameon=False, cmap='viridis')
elif 'total_counts' in adata.obs.columns:
    sc.pl.umap(adata, color='total_counts', ax=axes[1, 1], show=False,
               title='Total Counts', frameon=False, cmap='viridis')
else:
    axes[1, 1].text(0.5, 0.5, 'QC metric not found', 
                    ha='center', va='center', fontsize=14)
    axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_UMAP_overview.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ UMAP overview saved")

In [ ]:
# Quick statistics
n_clusters = adata.obs[CLUSTER_KEY].nunique()
n_tissues = adata.obs[TISSUE_KEY].nunique()
n_batches = adata.obs[BATCH_KEY].nunique()

print(f"Summary Statistics:")
print(f"  Total cells: {adata.n_obs:,}")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of tissues: {n_tissues}")
print(f"  Number of batches: {n_batches}")
print(f"\nCluster sizes:")
cluster_sizes = adata.obs[CLUSTER_KEY].value_counts().sort_index()
print(cluster_sizes)

### 1.2 GEP-Cluster Association Heatmap ⭐⭐⭐

**This is the most important plot for annotation!**

It shows which GEPs are enriched in which clusters, helping us:
1. Identify similar clusters (for potential merging)
2. Understand cluster "identity" through dominant GEPs
3. Link GEPs to biological functions

In [ ]:
# Extract GEP usage data
gep_cols = [f'GEP_{i}_{GEP_SUFFIX}' for i in range(1, N_GEPS + 1)]
gep_usage = adata.obs[gep_cols + [CLUSTER_KEY]].copy()

# Rename columns for cleaner display
gep_usage.columns = [f'GEP{i}' for i in range(1, N_GEPS + 1)] + ['cluster']

# Calculate mean usage per cluster
mean_usage = gep_usage.groupby('cluster').mean()

print(f"GEP-Cluster matrix shape: {mean_usage.shape}")
print(f"  {mean_usage.shape[0]} clusters × {mean_usage.shape[1]} GEPs")

# Save data
mean_usage.to_csv(f"{OUTPUT_DIR}/GEP_cluster_mean_usage.csv")
print(f"\n✓ Saved: GEP_cluster_mean_usage.csv")

In [ ]:
# Create hierarchical clustering heatmap
from scipy.cluster.hierarchy import linkage

# Calculate linkages
row_linkage = linkage(mean_usage.values, method='average', metric='euclidean')
col_linkage = linkage(mean_usage.T.values, method='average', metric='euclidean')

# Determine figure size based on cluster number
fig_width = max(12, N_GEPS * 0.6)
fig_height = max(10, n_clusters * 0.3)

# Create clustermap
g = sns.clustermap(
    mean_usage,
    row_linkage=row_linkage,
    col_linkage=col_linkage,
    cmap='RdYlBu_r',
    center=mean_usage.values.mean(),
    figsize=(fig_width, fig_height),
    cbar_kws={'label': 'Mean GEP Usage'},
    linewidths=0.5,
    linecolor='lightgray',
    yticklabels=True,
    xticklabels=True,
    dendrogram_ratio=0.15
)

g.ax_heatmap.set_xlabel('GEPs', fontsize=12, fontweight='bold')
g.ax_heatmap.set_ylabel('Clusters', fontsize=12, fontweight='bold')
plt.suptitle(f'GEP-Cluster Association Heatmap ({GEP_SUFFIX})', 
             y=0.98, fontsize=14, fontweight='bold')

plt.savefig(f"{OUTPUT_DIR}/02_GEP_cluster_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ GEP-Cluster heatmap saved")

In [ ]:
# Identify dominant GEPs for each cluster
TOP_N = 3

dominant_geps = []
for cluster in mean_usage.index.sort_values():
    cluster_usage = mean_usage.loc[cluster].sort_values(ascending=False)
    top_geps = cluster_usage.head(TOP_N)
    
    dominant_geps.append({
        'Cluster': cluster,
        'Top1_GEP': top_geps.index[0],
        'Top1_Usage': top_geps.values[0],
        'Top2_GEP': top_geps.index[1],
        'Top2_Usage': top_geps.values[1],
        'Top3_GEP': top_geps.index[2],
        'Top3_Usage': top_geps.values[2],
    })

dominant_df = pd.DataFrame(dominant_geps)
dominant_df.to_csv(f"{OUTPUT_DIR}/GEP_dominant_per_cluster.csv", index=False)

print("Top 3 dominant GEPs per cluster:")
print(dominant_df.to_string())
print(f"\n✓ Saved: GEP_dominant_per_cluster.csv")

### 1.3 Marker Gene Dotplot

Display key marker genes across clusters to identify cell types

In [ ]:
# Create a curated marker list for dotplot
# Focus on key discriminative markers

dotplot_markers = {
    'AT1': ['AGER', 'RTKN2', 'CAV1'],
    'AT2': ['SFTPC', 'SFTPB', 'LAMP3'],
    'AT2_prolif': ['STMN1', 'TOP2A', 'SFTPA2'],
    'Basal': ['TP63', 'KRT5', 'KRT15'],
    'Suprabasal': ['KRT6A', 'SERPINB4', 'KRT17'],
    'Ciliated': ['FOXJ1', 'RSPH1', 'PIFO'],
    'Deuterosomal': ['CCNO', 'CDC20B', 'FOXJ1'],
    'Club': ['SCGB1A1', 'SCGB3A2', 'CYP2F1'],
    'Goblet': ['MUC5AC', 'SPDEF', 'MUC5B'],
    'SMG': ['LYZ', 'LTF', 'AZGP1'],
    'Ionocyte': ['FOXI1', 'CFTR', 'ASCL3'],
    'Tuft': ['POU2F3', 'LRMP', 'KIT'],
    'NE': ['CHGA', 'GRP', 'ASCL1'],
    'Prolif': ['MKI67', 'TOP2A', 'CENPW'],
}

# Flatten and filter
dotplot_genes = []
for cell_type, genes in dotplot_markers.items():
    available = [g for g in genes if g in adata.var_names]
    dotplot_genes.extend(available)

dotplot_genes = list(dict.fromkeys(dotplot_genes))  # Remove duplicates, preserve order

print(f"Dotplot markers: {len(dotplot_genes)} genes")
print(f"Genes: {', '.join(dotplot_genes)}")

In [ ]:
# Create dotplot
fig_height = max(8, n_clusters * 0.25)
fig_width = max(10, len(dotplot_genes) * 0.3)

sc.pl.dotplot(
    adata,
    var_names=dotplot_genes,
    groupby=CLUSTER_KEY,
    dendrogram=True,
    figsize=(fig_width, fig_height),
    standard_scale='var',  # Normalize per gene
    save=f'_{OUTPUT_DIR}/03_marker_dotplot.png'
)

print("✓ Marker dotplot saved")

### 1.4 Key Marker UMAP

Visualize critical markers on UMAP to validate cluster identities

In [ ]:
# Select key discriminative markers
key_markers = [
    'TP63',    # Basal
    'SCGB1A1', # Secretory/Club
    'FOXJ1',   # Ciliated
    'MUC5AC',  # Goblet
    'SFTPC',   # AT2
    'AGER',    # AT1
    'MKI67',   # Proliferating
    'FOXI1',   # Ionocyte
]

# Filter available
key_markers = [m for m in key_markers if m in adata.var_names]

# Create multi-panel UMAP
sc.pl.umap(
    adata,
    color=key_markers,
    ncols=3,
    frameon=False,
    cmap='viridis',
    save=f'_{OUTPUT_DIR}/04_key_markers_umap.png'
)

print("✓ Key marker UMAP saved")

---
## 🔍 Round 2: Deep Dive Analysis

### 2.1 Identify Ambiguous Clusters

Find clusters that might represent:
- Transitional states (Basal → Secretory)
- Mixed populations
- Rare cell types

In [ ]:
# Calculate marker expression per cluster
# This helps identify which clusters express which markers

marker_categories = {
    'Basal': ['TP63', 'KRT5', 'KRT15'],
    'Secretory': ['SCGB1A1', 'SCGB3A2'],
    'Ciliated': ['FOXJ1', 'RSPH1'],
    'Goblet': ['MUC5AC', 'SPDEF'],
    'AT1': ['AGER', 'RTKN2'],
    'AT2': ['SFTPC', 'SFTPB'],
}

# Calculate mean expression per cluster for each category
cluster_signatures = {}

for cat_name, markers in marker_categories.items():
    available_markers = [m for m in markers if m in adata.var_names]
    if len(available_markers) == 0:
        continue
    
    # Get expression matrix (log-normalized)
    if 'log1p' in adata.layers:
        expr_data = pd.DataFrame(
            adata[:, available_markers].layers['log1p'].toarray() if hasattr(adata[:, available_markers].layers['log1p'], 'toarray') else adata[:, available_markers].layers['log1p'],
            index=adata.obs_names,
            columns=available_markers
        )
    else:
        expr_data = pd.DataFrame(
            adata[:, available_markers].X.toarray() if hasattr(adata[:, available_markers].X, 'toarray') else adata[:, available_markers].X,
            index=adata.obs_names,
            columns=available_markers
        )
    
    # Mean expression per cluster
    expr_data['cluster'] = adata.obs[CLUSTER_KEY].values
    mean_expr = expr_data.groupby('cluster').mean().mean(axis=1)  # Average across markers
    
    cluster_signatures[cat_name] = mean_expr

# Combine into dataframe
signature_df = pd.DataFrame(cluster_signatures)
signature_df.to_csv(f"{OUTPUT_DIR}/cluster_category_signatures.csv")

print("Cluster signatures (mean expression):")
print(signature_df.round(2))
print(f"\n✓ Saved: cluster_category_signatures.csv")

In [ ]:
# Visualize cluster signatures as heatmap
plt.figure(figsize=(10, max(8, n_clusters * 0.3)))

sns.heatmap(
    signature_df,
    cmap='RdYlBu_r',
    center=0,
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'Mean Log Expression'},
    linewidths=0.5
)

plt.title('Cluster Category Signatures', fontsize=14, fontweight='bold')
plt.xlabel('Cell Type Category', fontsize=12)
plt.ylabel('Cluster', fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_cluster_signatures_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Cluster signatures heatmap saved")

In [ ]:
# Identify ambiguous clusters (express multiple category markers)
# Normalize signatures to 0-1 range for comparison
signature_norm = (signature_df - signature_df.min()) / (signature_df.max() - signature_df.min())

# Find clusters with multiple high scores (>0.5)
threshold = 0.5
ambiguous_clusters = []

for cluster in signature_norm.index:
    high_scores = signature_norm.loc[cluster] > threshold
    if high_scores.sum() >= 2:  # At least 2 categories
        categories = ', '.join(signature_norm.loc[cluster][high_scores].index.tolist())
        ambiguous_clusters.append({
            'Cluster': cluster,
            'N_categories': high_scores.sum(),
            'Categories': categories,
            'Max_score': signature_norm.loc[cluster].max(),
        })

if len(ambiguous_clusters) > 0:
    ambiguous_df = pd.DataFrame(ambiguous_clusters)
    print("\n⚠️  Potentially ambiguous clusters (express multiple markers):")
    print(ambiguous_df.to_string(index=False))
    ambiguous_df.to_csv(f"{OUTPUT_DIR}/ambiguous_clusters.csv", index=False)
else:
    print("\n✓ No obvious ambiguous clusters detected")

### 2.2 GEP Top Genes Analysis

Examine which genes define each GEP to understand biological meaning

In [ ]:
# Load cNMF top genes (if available)
import glob

# Try to find the gene spectra file
cnmf_dir = "/home/h2048/data/py/1202/bbknn_celltype_analysis/Epithelial/output_complete_pipeline/cnmf_results"

# Look for harmony results
if GEP_SUFFIX == 'harmony':
    pattern = f"{cnmf_dir}/Epithelial_HarmonyBatchCorrected.gene_spectra_score.k_15.dt*.txt"
else:
    pattern = f"{cnmf_dir}/Epithelial_NoBatchCorrection.gene_spectra_score.k_15.dt*.txt"

spectra_files = glob.glob(pattern)

if len(spectra_files) > 0:
    spectra_file = spectra_files[0]
    print(f"Found spectra file: {spectra_file}")
    
    # Load gene spectra scores
    gene_spectra = pd.read_csv(spectra_file, sep='\t', index_col=0)
    
    print(f"\nGene spectra shape: {gene_spectra.shape}")
    print(f"Columns: {gene_spectra.columns.tolist()}")
    
    TOP_N_GENES = 30
    
    # For each GEP, get top genes
    gep_top_genes = {}
    for col in gene_spectra.columns:
        top_genes = gene_spectra[col].sort_values(ascending=False).head(TOP_N_GENES)
        gep_top_genes[col] = top_genes.index.tolist()
    
    # Save to file
    with open(f"{OUTPUT_DIR}/GEP_top_genes.txt", 'w') as f:
        for gep, genes in gep_top_genes.items():
            f.write(f"\n{'='*60}\n")
            f.write(f"{gep} - Top {TOP_N_GENES} Genes\n")
            f.write(f"{'='*60}\n")
            for i, gene in enumerate(genes, 1):
                f.write(f"{i:2d}. {gene}\n")
    
    print(f"\n✓ Saved: GEP_top_genes.txt")
    
    # Display first few GEPs
    for gep in list(gep_top_genes.keys())[:3]:
        print(f"\n{gep} top 10 genes:")
        print(', '.join(gep_top_genes[gep][:10]))
else:
    print("⚠️  cNMF gene spectra file not found")
    print(f"   Searched: {pattern}")

### 2.3 Tissue-Specific Analysis

Check if certain clusters are tissue-specific or pan-tissue

In [ ]:
# Create tissue × cluster contingency table
tissue_cluster = pd.crosstab(
    adata.obs[TISSUE_KEY],
    adata.obs[CLUSTER_KEY],
    normalize='columns'  # Show proportion within each cluster
)

# Visualize
plt.figure(figsize=(max(12, n_clusters * 0.4), 8))

sns.heatmap(
    tissue_cluster,
    cmap='YlOrRd',
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'Proportion'},
    linewidths=0.5
)

plt.title('Tissue Composition per Cluster', fontsize=14, fontweight='bold')
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Tissue', fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_tissue_cluster_composition.png", dpi=300, bbox_inches='tight')
plt.show()

tissue_cluster.to_csv(f"{OUTPUT_DIR}/tissue_cluster_composition.csv")
print("\n✓ Tissue-cluster composition saved")

In [ ]:
# Identify tissue-specific clusters (>80% from one tissue)
tissue_specific = []

for cluster in tissue_cluster.columns:
    max_tissue = tissue_cluster[cluster].idxmax()
    max_prop = tissue_cluster[cluster].max()
    
    if max_prop > 0.8:
        tissue_specific.append({
            'Cluster': cluster,
            'Dominant_Tissue': max_tissue,
            'Proportion': max_prop,
            'Specificity': 'High'
        })
    elif max_prop > 0.6:
        tissue_specific.append({
            'Cluster': cluster,
            'Dominant_Tissue': max_tissue,
            'Proportion': max_prop,
            'Specificity': 'Moderate'
        })

if len(tissue_specific) > 0:
    specific_df = pd.DataFrame(tissue_specific)
    print("\nTissue-specific clusters:")
    print(specific_df.to_string(index=False))
    specific_df.to_csv(f"{OUTPUT_DIR}/tissue_specific_clusters.csv", index=False)
else:
    print("\nNo highly tissue-specific clusters found (all are pan-tissue)")

---
## 💡 Round 3: Annotation Suggestions

### 3.1 Automated Cluster Annotation Suggestions

Based on marker expression + GEP analysis

In [ ]:
# Create annotation suggestions based on marker expression
annotation_suggestions = []

for cluster in signature_df.index:
    scores = signature_df.loc[cluster]
    top_category = scores.idxmax()
    top_score = scores.max()
    second_category = scores.nlargest(2).index[1]
    second_score = scores.nlargest(2).values[1]
    
    # Determine confidence
    if top_score > 1.5 and (top_score - second_score) > 0.5:
        confidence = 'High'
        suggestion = top_category
    elif top_score > 1.0:
        confidence = 'Moderate'
        if second_score > 0.8:
            suggestion = f"{top_category}/{second_category}"
        else:
            suggestion = top_category
    else:
        confidence = 'Low'
        suggestion = f"Unknown ({top_category}?)"
    
    # Get cluster size
    cluster_size = (adata.obs[CLUSTER_KEY] == cluster).sum()
    
    # Get dominant GEP
    dominant_gep = dominant_df[dominant_df['Cluster'] == cluster]['Top1_GEP'].values[0] if cluster in dominant_df['Cluster'].values else 'N/A'
    
    annotation_suggestions.append({
        'Cluster': cluster,
        'Size': cluster_size,
        'Percent': f"{cluster_size/adata.n_obs*100:.2f}%",
        'Suggestion': suggestion,
        'Confidence': confidence,
        'Top_Score': round(top_score, 2),
        'Dominant_GEP': dominant_gep,
    })

annotation_df = pd.DataFrame(annotation_suggestions)
annotation_df = annotation_df.sort_values('Size', ascending=False)

print("\n" + "="*80)
print("AUTOMATED ANNOTATION SUGGESTIONS")
print("="*80)
print(annotation_df.to_string(index=False))

annotation_df.to_csv(f"{OUTPUT_DIR}/annotation_suggestions.csv", index=False)
print(f"\n✓ Saved: annotation_suggestions.csv")

### 3.2 Export Cluster-wise Statistics for Manual Review

In [ ]:
# Comprehensive cluster statistics
cluster_stats = []

for cluster in adata.obs[CLUSTER_KEY].unique():
    cluster_cells = adata[adata.obs[CLUSTER_KEY] == cluster]
    
    stats_dict = {
        'Cluster': cluster,
        'N_cells': cluster_cells.n_obs,
        'Percent': f"{cluster_cells.n_obs/adata.n_obs*100:.2f}%",
    }
    
    # Tissue distribution
    tissue_dist = cluster_cells.obs[TISSUE_KEY].value_counts()
    for tissue in adata.obs[TISSUE_KEY].unique():
        stats_dict[f'Tissue_{tissue}'] = tissue_dist.get(tissue, 0)
    
    # Batch distribution
    batch_dist = cluster_cells.obs[BATCH_KEY].value_counts()
    stats_dict['N_batches'] = len(batch_dist)
    
    cluster_stats.append(stats_dict)

stats_df = pd.DataFrame(cluster_stats)
stats_df = stats_df.sort_values('N_cells', ascending=False)
stats_df.to_csv(f"{OUTPUT_DIR}/cluster_detailed_statistics.csv", index=False)

print("Cluster detailed statistics:")
print(stats_df.head(10))
print(f"\n✓ Saved: cluster_detailed_statistics.csv")

---
## 📝 Summary and Next Steps

### Generated Files:
1. `01_UMAP_overview.png` - Overall data structure
2. `02_GEP_cluster_heatmap.png` - ⭐ Main result for annotation
3. `03_marker_dotplot.png` - Marker gene expression
4. `04_key_markers_umap.png` - Spatial distribution of key markers
5. `05_cluster_signatures_heatmap.png` - Category-level signatures
6. `06_tissue_cluster_composition.png` - Tissue specificity
7. `annotation_suggestions.csv` - Automated suggestions
8. Multiple CSV files with quantitative data

### Recommended Workflow:
1. **Review GEP-Cluster heatmap** (file #2)
   - Identify clusters with similar GEP patterns → consider merging
   - Note dominant GEPs for each cluster

2. **Check marker dotplot** (file #3)
   - Validate cluster identities using known markers
   - Identify clear cell types (high confidence)

3. **Focus on ambiguous clusters**
   - Check `ambiguous_clusters.csv`
   - Review GEP top genes for biological interpretation

4. **Cross-tissue validation**
   - Use `tissue_cluster_composition.png`
   - Ensure annotations make sense across tissues

5. **Manual refinement**
   - Use `annotation_suggestions.csv` as starting point
   - Refine based on biological knowledge
   - Consider merging over-split clusters

### For Basal/Secretory/Goblet Disambiguation:
- Check specific markers: TP63 (Basal), SCGB1A1 (Secretory), MUC5AC (Goblet)
- Look for transitional states (co-expression patterns)
- Use GEP top genes to understand intermediate populations
- Consider tissue context (nasal vs bronchial secretory cells differ)


In [ ]:
print("\n" + "="*80)
print("ANNOTATION PIPELINE COMPLETE!")
print("="*80)
print(f"\nAll results saved to: {OUTPUT_DIR}")
print(f"\nKey files to review:")
print(f"  1. 02_GEP_cluster_heatmap.png")
print(f"  2. annotation_suggestions.csv")
print(f"  3. 03_marker_dotplot.png")
print(f"\nNext: Review these files and manually refine annotations!")